# Entrenamiento del detector de regiones (SmartPharma IA) — v2
Pasos:
1. **Entorno de ejecución → Cambiar tipo de entorno → GPU T4**.
2. En la carpeta de Drive (`BASE`) deben estar: `entrenar.py` (el nuevo) y `roboflow.zip` (export YOLOv8 de Roboflow).
3. Ejecuta las celdas en orden. Cambia solo los parámetros de la celda 4.

Novedades vs v1: **early stopping** (`--paciencia`), aumento de datos para texto, sobremuestreo de fotos con lote/fecha y mAP **por clase**.

In [ ]:
!nvidia-smi | head -n 12

In [ ]:
!pip -q install ultralytics pyyaml

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/9NO SEMESTRE/DISEÑO/ÁREAS/IA/IA_entrenamiento'
!cp "{BASE}/entrenar.py" /content/entrenar.py
!rm -rf /content/roboflow && mkdir -p /content/roboflow && unzip -q "{BASE}/roboflow.zip" -d /content/roboflow
!ls /content/roboflow

In [ ]:
# Parámetros. Ver "python entrenar.py --help".
#  --epocas 150 --paciencia 25 : corta solo si 25 épocas seguidas no mejoran; best.pt es la MEJOR época, no la última.
#  --sobremuestreo 3           : cada foto con lote/fecha entra 3 veces a train (cada copia con aumento distinto).
#  --arducam-desde 201         : solo las cajas >= 201 (Arducam) validan. Ajusta al número real.
#  --mapa mapa_cajas.csv       : solo si repetiste cajas con celular y Arducam.
!python /content/entrenar.py --datos /content/roboflow --salida /content/yolo     --modelo yolov8n.pt --epocas 150 --paciencia 25 --imgsz 960 --sobremuestreo 3 --arducam-desde 201

In [ ]:
# Curvas: mira que mAP50 de val deje de subir y la pérdida de val no suba (sobreajuste)
import pandas as pd
r = pd.read_csv('/content/yolo/runs/detector/results.csv'); r.columns = r.columns.str.strip()
print('épocas entrenadas:', len(r), '| mejor mAP50:', r['metrics/mAP50(B)'].max(), 'en época', int(r.loc[r['metrics/mAP50(B)'].idxmax(), 'epoch']))
r[['metrics/mAP50(B)', 'val/cls_loss']].plot(subplots=True, figsize=(8, 5))

In [ ]:
# Guarda en Drive en una carpeta NUEVA (no pisa resultado/ ni resultado_A/B)
!mkdir -p "{BASE}/resultado_v2"
!cp /content/yolo/runs/detector/weights/best.pt /content/yolo/runs/detector/weights/best.onnx "{BASE}/resultado_v2/"
!cp /content/yolo/runs/detector/results.csv "{BASE}/resultado_v2/"
!ls -la "{BASE}/resultado_v2"